In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler

# 1. LOAD DATA
print("--- Downloading and Loading GPS Trip Dataset ---")

try:
    path = kagglehub.dataset_download("ramakrishnanthiyagu/delivery-truck-trips-data")
    target_file = None
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith(('.xlsx', '.csv', '.xls')):
                target_file = os.path.join(root, file)
                break
        if target_file: break

    if not target_file: raise FileNotFoundError("No data files found.")

    if target_file.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(target_file, engine='openpyxl')
    else:
        df = pd.read_csv(target_file)

except Exception as e:
    print(f"!!! Error loading dataset: {e} !!!")
    raise

# Clean column names
df.columns = [str(c).strip() for c in df.columns]
print(f"--- Dataset Loaded: {len(df)} rows ---")

# 2. TIME & DELAY ENGINEERING
print("--- Engineering Delay & Time Features ---")
date_cols = ['trip_start_date', 'trip_end_date', 'actual_eta', 'Planned_ETA', 'BookingIDDate']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Safety for Delay Calculation
if 'actual_eta' in df.columns and 'Planned_ETA' in df.columns:
    df['delay_hours'] = (df['actual_eta'] - df['Planned_ETA']).dt.total_seconds() / 3600
else:
    df['delay_hours'] = 0

df['delay_hours'] = df['delay_hours'].fillna(0)
# Dynamic Delay Threshold
delay_threshold = df['delay_hours'].quantile(0.75)
df['delay_flag'] = (df['delay_hours'] > delay_threshold).astype(int) 

# Temporal Trend Awareness
df['month'] = df['trip_start_date'].dt.month
df['day_of_week'] = df['trip_start_date'].dt.dayofweek
df['is_peak_month'] = df['month'].isin([10, 11, 12]).astype(int)

# 3. DISTANCE & COST CLEANING
print("--- Cleaning Distance & Fuel Metrics ---")
df['TRANSPORTATION_DISTANCE_IN_KM'] = df['TRANSPORTATION_DISTANCE_IN_KM'].fillna(df['TRANSPORTATION_DISTANCE_IN_KM'].median())

FUEL_RATE = 87.67
MILEAGE = 3.5
df['est_fuel_cost'] = (df['TRANSPORTATION_DISTANCE_IN_KM'] / MILEAGE) * FUEL_RATE

# 4. VEHICLE LIFECYCLE (Logic 7.2.5 - Recency & Smoothing)
print("--- Tracking Vehicle Lifecycle (Advanced Smoothing) ---")
df = df.sort_values(by=['vehicle_no', 'trip_start_date'])

df['next_origin'] = df.groupby('vehicle_no')['Origin_Location'].shift(-1)
df['next_dest'] = df.groupby('vehicle_no')['Destination_Location'].shift(-1)
df['next_start_time'] = df.groupby('vehicle_no')['trip_start_date'].shift(-1)

# Lifecycle Flags
df['is_return_trip'] = (
    (df['Origin_Location'] == df['next_dest']) & 
    (df['Destination_Location'] == df['next_origin']) & 
    (df['Origin_Location'] != df['Destination_Location'])
).astype(int)

df['is_deadhead'] = (
    (df['Destination_Location'] != df['next_origin']) & 
    (df['next_origin'].notna()) & 
    (df['is_return_trip'] == 0)
).astype(int)

# Dwell time and Idle Loss
df['dwell_time_hrs'] = (df['next_start_time'] - df['trip_end_date']).dt.total_seconds() / 3600
df['dwell_time_hrs'] = df['dwell_time_hrs'].clip(lower=0).fillna(0)

# REFINED IDLE LOSS: Amplifies small dwell times for better signal strength
df['idle_loss_raw'] = np.log1p(df['dwell_time_hrs']) * (df['est_fuel_cost'] / 100)

# Past-Only Driver Performance
df['driver_delay_rate'] = df.groupby('vehicle_no')['delay_flag'].transform(lambda x: x.shift().expanding().mean())
df['driver_delay_rate'] = df['driver_delay_rate'].fillna(df['delay_flag'].mean())

# Continuity Threshold (90th Percentile)
threshold_90 = max(48, df['dwell_time_hrs'].quantile(0.90))
df['is_continuous'] = df['dwell_time_hrs'] < threshold_90

# 5. DEMAND & STABILITY (Recency Weighted)
print("--- Calculating Bayesian Risk & Recency-Aware Demand ---")

# REFINED LANE POPULARITY: Adds recency weighting to prefer active routes
df['trip_index'] = df.groupby(['Origin_Location', 'Destination_Location']).cumcount()
df['lane_popularity_raw'] = (
    df.groupby(['Origin_Location', 'Destination_Location'])['BookingID']
    .transform(lambda x: x.shift().expanding().count())
).fillna(0)

# Boost recent routes
df['lane_popularity_raw'] = (df['lane_popularity_raw'] * (1 + 0.05 * df['trip_index'])) + 1

# Time-Consistent Bayesian Smoothed Route Risk
df['route_risk_raw'] = (
    df.groupby(['Origin_Location', 'Destination_Location'])['delay_flag']
    .transform(lambda x: x.shift().expanding().std())
).fillna(0)

df['route_count'] = df.groupby(['Origin_Location', 'Destination_Location']).cumcount()
df['route_risk_smoothed'] = df['route_risk_raw'] * (df['route_count'] / (df['route_count'] + 10))

# Pre-normalization values
df['lane_popularity_log'] = np.log1p(df['lane_popularity_raw'])
df['cost_weight_raw'] = 1 / (1 + np.log1p(df['est_fuel_cost'] / 1000))

# Pre-calculate Distance Norm
max_dist = df['TRANSPORTATION_DISTANCE_IN_KM'].max()
df['dist_norm'] = df['TRANSPORTATION_DISTANCE_IN_KM'] / max_dist

# SCALING BUG FIX: Separate scaling to preserve semantic meaning of Confidence Score
scaler = MinMaxScaler()
scaled_cols = ['lane_popularity_log', 'cost_weight_raw', 'idle_loss_raw', 'route_risk_smoothed']
scaled_data = scaler.fit_transform(df[scaled_cols])

df[['lane_popularity', 'cost_weight', 'idle_loss', 'route_risk']] = scaled_data

# Recompute confidence AFTER scaling so that Confidence = 1 - Risk
df['confidence_score'] = 1 - df['route_risk']

# 6. RISK INTELLIGENCE & SCORING
print("--- Finalizing Decisions Layer (Logic 7.2.5) ---")

# Rebalanced Risk Weights
df['risk_score_raw'] = (
    0.15 * df['delay_flag'] + 
    0.20 * df['dist_norm'] + 
    0.30 * (1 - df['lane_popularity']) + 
    0.35 * df['route_risk']
)
risk_scaler = MinMaxScaler()
df['risk_score'] = risk_scaler.fit_transform(df[['risk_score_raw']])

LAMBDA_BASE = 0.062
GRACE_PERIOD = 4.0 

def calculate_logic_7_final(row):
    # Dynamic & Informed Fallback
    if pd.isna(row['next_start_time']) or not row['is_continuous']:
        fb_score = (0.3 + 
                    0.3 * row['lane_popularity'] + 
                    0.2 * row['cost_weight'] + 
                    0.2 * (1 - row['risk_score']))
        return round(np.clip(fb_score, 0, 1), 4)
    
    # Distance-Adjusted Lambda
    dist_scalar = np.log1p(row['TRANSPORTATION_DISTANCE_IN_KM'] / 100)
    adj_lambda = LAMBDA_BASE / max(1, dist_scalar)
    
    # Time Decay
    t_adj = max(0, row['dwell_time_hrs'] - GRACE_PERIOD)
    base_score = np.exp(-adj_lambda * t_adj)
    
    # Long-trip reward
    ops_score = base_score * (0.7 + 0.3 * row['dist_norm'])
    
    # Modifiers
    if row['is_return_trip']:
        ops_score = min(1.0, ops_score * 1.5) 
    elif row['is_deadhead']:
        ops_score = ops_score * 0.35 
        
    # REFINED SCORING: Positive stacking for better interpretability
    # 40% Ops, 20% Demand, 15% Cost, 15% Inverted Risk, 10% Reliability
    final_score = (ops_score * 0.40) + \
                  (row['lane_popularity'] * 0.20) + \
                  (row['cost_weight'] * 0.15) + \
                  ((1 - row['risk_score']) * 0.15) + \
                  ((1 - row['driver_delay_rate']) * 0.10)
    
    if row['is_peak_month']:
        final_score += 0.02

    return round(np.clip(final_score, 0, 1), 4)

df['efficiency_score'] = df.apply(calculate_logic_7_final, axis=1)

# Quantile-based labels
df['efficiency_label'] = pd.qcut(
    df['efficiency_score'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

# 7. EXPORT
required_columns = [
    'BookingID', 'vehicle_no', 'Origin_Location', 'Destination_Location', 
    'TRANSPORTATION_DISTANCE_IN_KM', 'day_of_week', 'month', 'delay_flag',
    'lane_popularity', 'risk_score', 'driver_delay_rate', 
    'route_risk', 'confidence_score', 'idle_loss', 'is_return_trip', 'is_deadhead', 
    'efficiency_score', 'efficiency_label'
]

if 'Market/Regular' in df.columns:
    required_columns.append('Market/Regular')

final_df = df[required_columns].copy()
final_df.to_csv("freightiq_gps_ready_data_v7.csv", index=False)

print("--- SUCCESS! Logic 7.2.5 Fully Optimized System Generated ---")
print(f"Total trips: {len(final_df)}")
print("\n--- Summary Stats ---")
print(final_df[['efficiency_score', 'lane_popularity', 'risk_score', 'confidence_score']].describe().loc[['mean', 'min', 'max']])

--- Downloading and Loading GPS Trip Dataset ---
--- Dataset Loaded: 6880 rows ---
--- Engineering Delay & Time Features ---
--- Cleaning Distance & Fuel Metrics ---
--- Tracking Vehicle Lifecycle (Advanced Smoothing) ---
--- Calculating Bayesian Risk & Recency-Aware Demand ---
--- Finalizing Decisions Layer (Logic 7.2.5) ---
--- SUCCESS! Logic 7.2.5 Fully Optimized System Generated ---
Total trips: 6880

--- Summary Stats ---
      efficiency_score  lane_popularity  risk_score  confidence_score
mean          0.439356         0.310593    0.478903          0.713125
min           0.040300         0.000000    0.000000          0.000000
max           0.968500         1.000000    1.000000          1.000000


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# 1. LOAD THE OPTIMIZED DATASET
print("--- Loading Logic 7.2.5 Intelligence Dataset ---")
try:
    df = pd.read_csv("freightiq_gps_ready_data_v7.csv")
except FileNotFoundError:
    print("!!! Error: 'freightiq_gps_ready_data_v7.csv' not found. Please run your processor script first. !!!")
    exit()

# 2. PREPROCESSING
print("--- Preparing Features for Training ---")

# Handle Market/Regular Encoding (Converting to binary)
if 'Market/Regular' in df.columns:
    df['is_regular'] = df['Market/Regular'].apply(lambda x: 1 if str(x).strip() == 'Regular' else 0)
else:
    df['is_regular'] = 1

# Drop rows where target or critical features are missing
df = df.dropna(subset=['efficiency_score', 'TRANSPORTATION_DISTANCE_IN_KM'])

# 3. FEATURE SELECTION (The 5 Layers of Intelligence)
# We exclude IDs and raw text strings, focusing on the numerical/engineered signals
features = [
    'TRANSPORTATION_DISTANCE_IN_KM', 
    'day_of_week', 
    'month', 
    'delay_flag',
    'lane_popularity', 
    'risk_score', 
    'driver_delay_rate', 
    'route_risk',
    'is_return_trip',
    'is_deadhead',
    'is_regular'
]

X = df[features]
y = df['efficiency_score']

# 4. TRAIN-TEST SPLIT (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. HYPERPARAMETER TUNING
print("--- Tuning XGBoost Regressor (Logic 7.2.5 Optimized) ---")
# Using a grid that balances depth with regularization to handle noise
param_grid = {
    'n_estimators': [500, 1000, 1500],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
    'gamma': [0.1, 0.2],
    'reg_alpha': [0.1, 0.5], # L1 regularization
    'reg_lambda': [1.5, 2.0]  # L2 regularization
}

xgb_reg = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

# Randomized Search for efficient tuning
search = RandomizedSearchCV(
    xgb_reg, 
    param_distributions=param_grid, 
    n_iter=15, 
    scoring='r2', 
    cv=3, 
    verbose=1, 
    random_state=42
)

search.fit(X_train, y_train)
best_model = search.best_estimator_

# 6. EVALUATION
print("\n--- Final Model Evaluation ---")
preds = best_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"Final RMSE (Error Margin): {rmse:.4f}")
print(f"Final R-squared Score (Accuracy): {r2:.4f}")

# 7. SAVE THE ENGINE & FEATURE LIST
# This model will be used by app.py for real-time pricing
print("\n--- Saving FreightIQ Intelligence Engine ---")
joblib.dump(best_model, "freightiq_ai_engine_v7.pkl")
joblib.dump(features, "v7_feature_list.pkl")

# 8. FEATURE IMPORTANCE (Identifying Pricing Drivers)
importance = pd.DataFrame({
    'feature': features,
    'importance': best_model.feature_importances_
}).sort_values(by='importance', ascending=False)

print("\nTop 5 Drivers of Logistics Efficiency:")
print(importance.head(5))

print("\n--- SUCCESS! Your AI Engine is ready. ---")

--- Loading Logic 7.2.5 Intelligence Dataset ---
--- Preparing Features for Training ---
--- Tuning XGBoost Regressor (Logic 7.2.5 Optimized) ---
Fitting 3 folds for each of 15 candidates, totalling 45 fits

--- Final Model Evaluation ---
Final RMSE (Error Margin): 0.0922
Final R-squared Score (Accuracy): 0.6044

--- Saving FreightIQ Intelligence Engine ---

Top 5 Drivers of Logistics Efficiency:
                         feature  importance
9                    is_deadhead    0.606156
3                     delay_flag    0.075024
4                lane_popularity    0.055505
5                     risk_score    0.050855
0  TRANSPORTATION_DISTANCE_IN_KM    0.049481

--- SUCCESS! Your AI Engine is ready. ---


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score, roc_auc_score
import joblib

# 1. LOAD THE DATA
print("--- Loading Intelligence Dataset ---")
df = pd.read_csv("freightiq_gps_ready_data_v7.csv")
df = df.dropna(subset=['TRANSPORTATION_DISTANCE_IN_KM'])

# Feature subset for both models
features = [
    'TRANSPORTATION_DISTANCE_IN_KM', 'day_of_week', 'month',
    'lane_popularity', 'route_risk', 'driver_delay_rate',
    'is_return_trip', 'is_deadhead'
]

# ==========================================
# STAGE 1: THE RISK ENGINE (Classifier)
# Purpose: Predict the probability of Delay
# ==========================================
print("\n--- STAGE 1: Training Delay Predictor (Risk Engine) ---")
X_cls = df[features]
y_cls = df['delay_flag']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

risk_engine = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    objective='binary:logistic',
    random_state=42
)

risk_engine.fit(X_train_c, y_train_c)

# Calculate Probability for the entire dataset
# This is the "ML Intelligence" layer
df['predicted_delay_prob'] = risk_engine.predict_proba(X_cls)[:, 1]

print(f"Risk Engine Accuracy: {accuracy_score(y_test_c, risk_engine.predict(X_test_c)):.4f}")
print(f"Risk Engine AUC Score: {roc_auc_score(y_test_c, risk_engine.predict_proba(X_test_c)[:, 1]):.4f}")

# ==========================================
# STAGE 2: THE PRICING ENGINE (Regressor)
# Purpose: Calculate Probabilistic Efficiency
# ==========================================
print("\n--- STAGE 2: Training Pricing Engine (Elite Layer) ---")

# RE-CALCULATING TARGET (Logic 8.0 - Probabilistic)
# Inverting Predicted Probability: High Delay Prob = Lower Efficiency
def calculate_elite_score(row):
    # Base Operational Efficiency (Ops)
    ops_score = row['efficiency_score'] # This was our old historical target
    
    # NEW ELITE FORMULA
    # 40% Ops + 20% Demand + 15% Cost + 15% Delay-Freedom + 10% Reliability
    final_score = (
        0.40 * ops_score +                      # Operation Logic
        0.20 * row['lane_popularity'] +         # Demand Logic
        0.15 * (1 - row['risk_score']) +        # Route Risk Logic
        0.15 * (1 - row['predicted_delay_prob']) + # ML Predicted Delay Prob (Forward Looking)
        0.10 * (1 - row['driver_delay_rate'])   # Driver Reliability
    )
    return round(np.clip(final_score, 0, 1), 4)

df['probabilistic_efficiency'] = df.apply(calculate_elite_score, axis=1)

# Training the final Regressor on this new Elite target
X_reg = df[features + ['predicted_delay_prob']]
y_reg = df['probabilistic_efficiency']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

pricing_engine = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.03,
    objective='reg:squarederror',
    random_state=42
)

pricing_engine.fit(X_train_r, y_train_r)

r2 = r2_score(y_test_r, pricing_engine.predict(X_test_r))
print(f"Final Elite R2 Score: {r2:.4f}")

# ==========================================
# 3. SAVE THE ELITE STACK
# ==========================================
print("\n--- Saving Elite AI Stack ---")
joblib.dump(risk_engine, "freightiq_risk_engine_v8.pkl")
joblib.dump(pricing_engine, "freightiq_pricing_engine_v8.pkl")
joblib.dump(features + ['predicted_delay_prob'], "v8_feature_list.pkl")

print("\nSuccess! Your system now predicts the future before pricing it.")

--- Loading Intelligence Dataset ---

--- STAGE 1: Training Delay Predictor (Risk Engine) ---
Risk Engine Accuracy: 0.8336
Risk Engine AUC Score: 0.8801

--- STAGE 2: Training Pricing Engine (Elite Layer) ---
Final Elite R2 Score: 0.9121

--- Saving Elite AI Stack ---

Success! Your system now predicts the future before pricing it.
